In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table, Column
from astropy.table import join
from astropy.table import vstack
from pathlib import Path
import astropy.io.ascii as ascii

from lsst.daf.butler import Butler
from pfs.datamodel import TargetType
from pfs.datamodel import FiberStatus
%matplotlib inline

from pfs.datamodel.pfsTargetSpectra import PfsTargetSpectra
from pfs.datamodel import PfsCoZCandidates, PfsZCandidates

from IPython.display import clear_output
from scipy.ndimage import uniform_filter1d
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import median_filter
from scipy import ndimage

from collections import defaultdict

In [11]:
# List of tracts for the November run

import numpy as np
import pandas as pd

upper_spring = [10043, 9812, 9570, 9315, 9073]
lower_spring = [10040, 9798, 9556, 9313, 9070]

upper_autumn = [9981, 9738, 9479, 9253, 9011, 8769, 9957, 9715, 10184, 9942, 9700]
lower_autumn = [9964, 9722, 9469, 9249, 9007, 8765, 9955, 9713, 10181, 9939, 9697]

index = np.hstack([
    np.array(lower_autumn).reshape(-1, 1),
    np.array(upper_autumn).reshape(-1, 1)
])

tracts = []

for i, j in index:
    tracts.extend(range(i, j + 1))

df = pd.DataFrame({"tract": tracts})

df.to_csv("../dat/november_tracts_autumn.csv", index=False)

## Get target selection catalog from HSC

In [2]:
Tracts = ascii.read("../dat/november_tracts.csv")['tract']

hdu = fits.open("/lustre/work/YukaYamada/data/PFS/s23b_wide/ssp_co_targets/s23b_spring.fits")
data = hdu[1].data

mask = np.isin(data['tract'], Tracts)

spring_selected = data[mask]

hdu = fits.open("/lustre/work/YukaYamada/data/PFS/s23b_wide/ssp_co_targets/s23b_autumn.fits")
data = hdu[1].data

mask = np.isin(data['tract'], Tracts)

autumn_selected = data[mask]

table1 = Table(spring_selected)
table2 = Table(autumn_selected)

target_catalog = vstack([table1, table2])

## Apply updated bright stellar mask

In [3]:
path = '/lustre/work/YukaYamada/data/PFS/s23b-bsmask/tracts/galaxies/'
tables = []

for tract in Tracts:
    filename = path + f'{tract}.fits'
    if (not os.path.exists(filename)):
        print(f"file not available for tract {tract}")
        continue

    with fits.open(filename) as hdu:
        tables.append(Table(hdu[1].data))

bsmask = vstack(tables)
bsmask.remove_columns(['ra', 'dec'])

merged_catalog = join(
    target_catalog,
    bsmask,
    keys_left='object_id',
    keys_right='id',
    join_type='left'
)

merged_catalog.remove_columns(['id'])

file not available for tract 9981
file not available for tract 9011


## Get imaging weights

In [4]:
import healpy as hp

nside = 256
target_ra = merged_catalog['ra']
target_dec = merged_catalog['dec']
healpix = hp.ang2pix(nside, target_ra, target_dec, nest=False, lonlat=True)
merged_catalog['healpix'] = healpix

filename = '/home/YukaYamada/repository/PFS/PFS_imaging/src/pfsimaging/dat/nn_weights.fits'
hdu = fits.open(filename)
data = hdu[1].data
hdu.close()

merged_catalog = join(
    merged_catalog,
    Table(data),
    keys='healpix', 
    join_type='left'
)

## Get fiber assignment weights

In [36]:
#We will get
#1. Which targets were assigned a fiber
#2. Which targets were "potentially assigned targets"
#3. Which targets lies in 2 Visits field (For November Run)

## Cross match with PFS data

In [ ]:
##All fiber assigned objects

##Observed objects with Visits, exposure time and fiberID

## Lam1D results (redshift, [OII]S/N)

In [10]:
#Get all of the visits and exposure time in the November run
catId = 10091 # catId for SSP Cosmology scientific targets (note: this includes ancillary targets as well)

# SP
#pfs_base_dir = "/shared/pfs/programs/"
# idark
pfs_base_dir = "/lustre/work/jingjing.shi/PFS_SSP/hscpfs.mtk.nao.ac.jp/fileaccess/pfs/programs/"

# run25
semester_code = 'S25B-OT02'
collections = ['run25_February2026'] # release collection
combination = 'selected_run25' 

# S25A April2026
#semester_code = 'S25A-OT02'
#collections = ['S25A_April2026']
#combination = 'selected_S25A'

release = collections[0]

butler = Butler(pfs_base_dir + semester_code + '/2d/', collections=collections)
dataRef = list(butler.registry.queryDatasets('pfsCoadd'))

ogm = butler.get("objectGroupMap", combination=combination, cat_id=catId)

# note: here the objId are science fibers that are marked as good fibers
objid_release = ogm.objId
groupid_release = ogm.objGroup
nobj_release = len(objid_release)

# get the unique groupid_release and rank them by ascending order of the value
unique_groupids = np.unique(groupid_release)

path = "../dat/obslog/processed/"

# for S25B November run
obslog_file = "co_nov2025.csv" 
obslog = Table.read(path + obslog_file)

# here we combine the obslog of march, may, and june together for S25A
#obslog = vstack([obslog_march, obslog_may, obslog_june])

ppc_name_obslog = np.unique(obslog['sequence_name'])
print(f'obslog: {len(ppc_name_obslog)} pointings observed.')

# map ppc_name to visit_id list in obslog
dict_obslog_visits = {}
for ppc_name in ppc_name_obslog:
    dict_obslog_visits[ppc_name] = obslog['visit_id'][obslog['sequence_name'] == ppc_name]

all_visits = obslog['visit_id']
visit_to_exptime = {
    visit: (
        float(exp),
        float(eet_b),
        float(eet_r),
        float(eet_n)
    )
    for visit, exp, eet_b, eet_r, eet_n in zip(
        obslog["visit_id"],
        obslog["avg_exptime"],
        obslog["eet_b"],
        obslog["eet_r"],
        obslog["eet_n"]
    )
}

obslog: 84 pointings observed.


/var/tmp/pbs.575468.idark/ipykernel_169836/451672102.py:57: UserWarning: Warning: converting a masked element to nan.
  float(eet_n)


In [8]:
#merged_catalog: all catalog
#applied_catalog: all targets that were applied a fiber
#observed_catalog: alltargets that have successful measurement for all visits

obj_to_visits = defaultdict(list)
obj_to_fibers = defaultdict(list)
obj_to_avg_exptime = defaultdict(list)
obj_to_eet_b = defaultdict(list)
obj_to_eet_r = defaultdict(list)
obj_to_eet_n = defaultdict(list)

all_good = []
all_bad = []

for visit in all_visits:
    pfsConfig = butler.get('pfsConfig', visit=visit)

    good = pfsConfig.select(
        targetType=TargetType.SCIENCE,
        fiberStatus=FiberStatus.GOOD
    )

    bad = pfsConfig.select(
        targetType=TargetType.SCIENCE,
        fiberStatus=~FiberStatus.GOOD
    )

    # good / bad ID を保存
    all_good.extend(good.objId)
    all_bad.extend(bad.objId)

    # exposure 情報
    exp, eet_b, eet_r, eet_n = visit_to_exptime[visit]

    # good + bad science targets
    obj_ids = np.concatenate([good.objId, bad.objId])
    fiber_ids = np.concatenate([good.fiberId, bad.fiberId])

    for obj_id, fiber_id in zip(obj_ids, fiber_ids):
        obj_to_visits[obj_id].append(int(visit))
        obj_to_fibers[obj_id].append(int(fiber_id))

        obj_to_avg_exptime[obj_id].append(exp)
        obj_to_eet_b[obj_id].append(eet_b)
        obj_to_eet_r[obj_id].append(eet_r)
        obj_to_eet_n[obj_id].append(eet_n)

goodIDs = np.unique(np.array(all_good, dtype=np.int64))
badIDs = np.unique(np.array(all_bad, dtype=np.int64))
allIDs = np.unique(np.concatenate([goodIDs, badIDs]))

merged_catalog['applied'] = np.isin(
    merged_catalog['object_id'],
    allIDs
)

applied_catalog = merged_catalog[merged_catalog['applied']].copy()
applied_catalog['BadFiber_flag'] = ~np.isin(applied_catalog['object_id'], badIDs)

objids = applied_catalog['object_id']

applied_catalog['visitID'] = [
    obj_to_visits[objid] for objid in objids
]

applied_catalog['fiberID'] = [
    obj_to_fibers[objid] for objid in objids
]

applied_catalog['avg_exptime'] = [
    obj_to_avg_exptime[objid] for objid in objids
]

applied_catalog['eet_b'] = [
    obj_to_eet_b[objid] for objid in objids
]

applied_catalog['eet_r'] = [
    obj_to_eet_r[objid] for objid in objids
]

applied_catalog['eet_n'] = [
    obj_to_eet_n[objid] for objid in objids
]

observed_catalog = applied_catalog[applied_catalog['BadFiber_flag']]

In [17]:
## Merge Lam1D results and get redshift, [OII]S/N for the observed catalogs

from astropy.io import fits
from astropy.table import Table, join, vstack
import numpy as np

config_1drp = "modified"

tables = []

for gid in unique_groupids:
    fn_lam1d = (
        f"{pfs_base_dir}/{semester_code}/lam1d/{release}/"
        f"{config_1drp}/10091_{gid}/data/pfsCoZcandidates-10091.fits"
    )

    with fits.open(fn_lam1d) as hdul:
        lam1d_basic = Table(hdul["TARGET"].data)
        lam1d_galaxy = Table(hdul["GALAXY_CANDIDATES"].data)
        lam1d_lines = Table(hdul["GALAXY_LINES"].data)

    # TARGET: objectID, targetID, ra, dec
    basic = lam1d_basic["objId", "targetId", "ra", "dec"]
    basic.rename_column("objId", "object_id")

    # GALAXY_CANDIDATES: use best solution only
    galaxy = lam1d_galaxy[lam1d_galaxy["cRank"] == 0]
    galaxy = galaxy["targetId", "redshift"]

    combined = join(
        basic,
        galaxy,
        keys="targetId",
        join_type="left"
    )

    # [OII] lines
    oii3726 = lam1d_lines[lam1d_lines["lineName"] == "[OII]3726"]
    oii3729 = lam1d_lines[lam1d_lines["lineName"] == "[OII]3729"]

    oii3726 = Table(oii3726)
    oii3729 = Table(oii3729)

    for col in oii3726.colnames:
        if col != "targetId":
            oii3726.rename_column(col, f"{col}_OII3726")

    for col in oii3729.colnames:
        if col != "targetId":
            oii3729.rename_column(col, f"{col}_OII3729")

    combined = join(combined, oii3726, keys="targetId", join_type="left")
    combined = join(combined, oii3729, keys="targetId", join_type="left")

    tables.append(combined)

lam1d_oii_table = vstack(tables, join_type="outer")

lam1d_oii_table["SNR_OII"] = flux_oii / fluxerr_oii

final_table = lam1d_oii_table[
    "object_id",
    "redshift",
    "lineFlux_OII3726",
    "lineFlux_OII3729",
    "lineFluxError_OII3726",
    "lineFluxError_OII3729"
]

observed_catalog = join(
    observed_catalog,
    final_table,
    keys='object_id', 
    join_type='left'
)

## Redshift success rate weights